# 03 — Deep learning models from scratch: BiLSTM, TextCNN and Attention BiLSTM

Three vocabulary-based classifiers on the cached lemmatized Persian text:

- **BiLSTM** — the deployed baseline, ported from `training/train_lstm.py`;
- **TextCNN** — parallel 1D-convolution n-gram detectors with a length-masked max-pool;
- **Attention BiLSTM** — BiLSTM encoder pooled by additive attention over real tokens.

Every section follows the same **load-or-train** pattern: when the checkpoint
exists under `models/` (and `FORCE_RETRAIN` is off) it is rebuilt from the
config stored inside it and loaded strict-ly; otherwise the model is trained
on CPU and saved in the shared checkpoint format (`state_dict` + `vocab` +
`config`) that `aitextd/loaders.py` consumes at serving time.

Evaluation: the canonical v1 train/test split, then one v2 cross-domain
condition as a generalization sanity check.

Runtime: **CPU is enough** (~15 minutes when all three models train).


In [ ]:
import sys
from pathlib import Path

# Make `aitextd` importable regardless of the notebook working directory.
ROOT_HINT = Path.cwd()
while ROOT_HINT != ROOT_HINT.parent and not (ROOT_HINT / "aitextd" / "paths.py").exists():
    ROOT_HINT = ROOT_HINT.parent
sys.path.insert(0, str(ROOT_HINT))

from aitextd.paths import PROJECT_ROOT

FORCE_RETRAIN = False  # True: retrain all three models and rewrite their models.json entries

# Data lives under data/ (the tokenized cache and split indices were moved
# here from the old training/ location).
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_JSON = PROJECT_ROOT / "models.json"

DATASET_CSV = DATA_DIR / "processedDataset.csv"
TOKENIZED_JOBLIB = DATA_DIR / "tokenized_texts.joblib"
SPLIT_JOBLIB = DATA_DIR / "split_indices.joblib"
DATASET_V2_PARQUET = DATA_DIR / "dataset_v2.parquet"
SPLITS_V2_JOBLIB = DATA_DIR / "splits_v2.joblib"

RANDOM_SEED = 42
DEVICE = "cpu"


In [ ]:
import joblib
import pandas as pd

df = pd.read_csv(DATASET_CSV)
tokenized = joblib.load(TOKENIZED_JOBLIB)
split = joblib.load(SPLIT_JOBLIB)
train_idx, test_idx = split["train_idx"], split["test_idx"]
assert len(tokenized) == len(df)

y = df["aiGenerated"].to_numpy()
train_tokens = [tokenized[i] for i in train_idx]
test_tokens = [tokenized[i] for i in test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(f"{len(df)} documents | train {len(train_idx)} / test {len(test_idx)}")


## 1) Compact metric helper

Condensed from `training/eval_metrics.py`: accuracy, precision / recall / F1,
ROC-AUC and expected calibration error (15 equal-width bins).


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

ECE_BINS = 15


def expected_calibration_error(y_true, probs, bins=ECE_BINS):
    """ECE = sum_b (n_b / N) * |accuracy_b - confidence_b| over equal-width bins."""
    y_true, probs = np.asarray(y_true, dtype=float), np.asarray(probs, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    edges[0], edges[-1] = -np.inf, np.inf
    ece, n = 0.0, len(y_true)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (probs > lo) & (probs <= hi)
        if mask.any():
            ece += mask.sum() / n * abs(y_true[mask].mean() - probs[mask].mean())
    return float(ece)


def evaluate(y_true, probs, threshold=0.5):
    """Binary metric bundle for human(=0) vs AI(=1) at the given threshold."""
    y_true, probs = np.asarray(y_true), np.asarray(probs, dtype=float)
    pred = (probs >= threshold).astype(int)
    confusion = {
        "tn": int(((pred == 0) & (y_true == 0)).sum()),
        "fp": int(((pred == 1) & (y_true == 0)).sum()),
        "fn": int(((pred == 0) & (y_true == 1)).sum()),
        "tp": int(((pred == 1) & (y_true == 1)).sum()),
    }
    return {
        "accuracy": round(float(accuracy_score(y_true, pred)), 4),
        "precision": round(float(precision_score(y_true, pred)), 4),
        "recall": round(float(recall_score(y_true, pred)), 4),
        "f1": round(float(f1_score(y_true, pred)), 4),
        "roc_auc": round(float(roc_auc_score(y_true, probs)), 4),
        "ece": round(expected_calibration_error(y_true, probs), 4),
        "confusion": confusion,
    }


## 2) Shared training machinery

Vocabulary and padding follow `training/train_lstm.py` exactly: `<pad>`=0 /
`<unk>`=1, words with frequency >= 2 up to 30k entries, sequences
truncated/padded to 200 tokens. The training loop is the project-wide pattern:
`BCEWithLogitsLoss` + Adam, per-epoch monitoring on the held-out test split,
keeping the best-F1 epoch's weights.


In [ ]:
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from aitextd.architectures import (
    AttentionBiLSTMClassifier,
    BiLSTMClassifier,
    TextCNNClassifier,
)

MAX_VOCAB_SIZE = 30_000
MIN_FREQ = 2
MAX_LEN = 200
PAD_IDX, UNK_IDX = 0, 1
BATCH_SIZE = 64
LR = 1e-3

torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(4)


class TextDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences, self.labels = sequences, labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def build_vocab(train_tokens):
    """<pad>=0 / <unk>=1, then words by descending frequency (>= MIN_FREQ) up
    to MAX_VOCAB_SIZE — identical to training/train_lstm.py."""
    counter = Counter()
    for tokens in train_tokens:
        counter.update(tokens)
    vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
    for word, freq in counter.most_common(MAX_VOCAB_SIZE):
        if freq < MIN_FREQ:
            break
        if word not in vocab:
            vocab[word] = len(vocab)
    return vocab


def encode(tokens, vocab, max_len=MAX_LEN):
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    return ids + [PAD_IDX] * (max_len - len(ids))


def encode_split(tokens_split, vocab):
    return torch.tensor([encode(t, vocab) for t in tokens_split], dtype=torch.long)


def predict_probs(model, X, batch_size=256):
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            probs.append(torch.sigmoid(model(X[i : i + batch_size])).numpy())
    return np.concatenate(probs)


def train_model(model, X_train, y_train, X_eval, y_eval, epochs, lr=LR):
    """BCEWithLogitsLoss + Adam; per-epoch monitoring on the held-out split,
    best-F1 weights kept (the train_lstm.py pattern)."""
    loader = DataLoader(
        TextDataset(X_train, torch.tensor(y_train, dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=True,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    best_f1, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
        probs = predict_probs(model, X_eval)
        epoch_f1 = f1_score(y_eval, (probs >= 0.5).astype(int))
        print(f"epoch {epoch}/{epochs}  loss={total_loss / len(X_train):.4f}  test_f1={epoch_f1:.4f}")
        if epoch_f1 > best_f1:
            best_f1 = epoch_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    return model


def save_checkpoint(path, model, vocab, config):
    """Shared format: state_dict + vocab + config dict, so aitextd/loaders.py
    can rebuild the model offline at serving time."""
    torch.save({"state_dict": model.state_dict(), "vocab": vocab, "config": config}, path)
    print("saved", path)


def load_or_train(make_model, extra_config, filename, epochs, model_id):
    """Load-or-train one model: rebuild strict-ly from the checkpoint's stored
    config when present (and FORCE_RETRAIN is False), else train a fresh
    vocabulary + model and save in the shared format."""
    path = MODELS_DIR / filename
    if path.exists() and not FORCE_RETRAIN:
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
        vocab, config = ckpt["vocab"], ckpt["config"]
        model = make_model(len(vocab), config)
        model.load_state_dict(ckpt["state_dict"])  # strict by default
        model.eval()
        print(f"loaded checkpoint {path.name}")
    else:
        vocab = build_vocab(train_tokens)
        config = {"max_len": MAX_LEN, "pad_idx": PAD_IDX, "unk_idx": UNK_IDX, **extra_config}
        model = make_model(len(vocab), config)
        print(f"training {model_id} (vocab {len(vocab)}) ...")
        model = train_model(
            model,
            encode_split(train_tokens, vocab), y_train,
            encode_split(test_tokens, vocab), y_test,
            epochs,
        )
        save_checkpoint(path, model, vocab, config)
    return model, vocab, config


## 3) Model 1 — BiLSTM (deployed baseline, ported from `training/train_lstm.py`)

Bidirectional LSTM over learned 128-d embeddings; the final hidden states of
both directions are concatenated and classified by a linear head. Trained for
5 epochs, matching the shipped checkpoint.


In [ ]:
def make_bilstm(vocab_size, config):
    return BiLSTMClassifier(
        vocab_size, config["embed_dim"], config["hidden_dim"], pad_idx=config["pad_idx"]
    )


bilstm, bilstm_vocab, bilstm_config = load_or_train(
    make_bilstm, {"embed_dim": 128, "hidden_dim": 128},
    "lstmModel.pt", epochs=5, model_id="BiLSTM",
)
bilstm_probs = predict_probs(bilstm, encode_split(test_tokens, bilstm_vocab))
bilstm_metrics = evaluate(y_test, bilstm_probs)
print("BiLSTM v1 test:", bilstm_metrics)


## 4) Model 2 — TextCNN

Parallel 1D convolutions (kernel sizes 3, 4, 5; 100 feature maps each) over
the embeddings act as n-gram detectors; a length mask keeps windows of pure
padding out of each kernel's global max-pool. The fastest of the three to
train (8 epochs on CPU in a few minutes).


In [ ]:
def make_textcnn(vocab_size, config):
    return TextCNNClassifier(
        vocab_size, config["embed_dim"], config["n_filters"],
        tuple(config["kernel_sizes"]), pad_idx=config["pad_idx"], dropout=config["dropout"],
    )


textcnn, textcnn_vocab, textcnn_config = load_or_train(
    make_textcnn,
    {"embed_dim": 128, "n_filters": 100, "kernel_sizes": [3, 4, 5], "dropout": 0.3},
    "textcnnModel.pt", epochs=8, model_id="TextCNN",
)
textcnn_probs = predict_probs(textcnn, encode_split(test_tokens, textcnn_vocab))
textcnn_metrics = evaluate(y_test, textcnn_probs)
print("TextCNN v1 test:", textcnn_metrics)


## 5) Model 3 — Attention BiLSTM

Same BiLSTM encoder, but pooled by **additive attention**: a learned score
vector scores every hidden state and the softmax is masked over real tokens,
so the summary is built only from actual words (the baseline keeps only the
last hidden state per direction). Trained for 4 epochs.


In [ ]:
def make_attention_bilstm(vocab_size, config):
    return AttentionBiLSTMClassifier(
        vocab_size, config["embed_dim"], config["hidden_dim"],
        pad_idx=config["pad_idx"], dropout=config["dropout"],
    )


attention_bilstm, attn_vocab, attn_config = load_or_train(
    make_attention_bilstm, {"embed_dim": 128, "hidden_dim": 128, "dropout": 0.3},
    "attnBilstmModel.pt", epochs=4, model_id="Attention BiLSTM",
)
attn_probs = predict_probs(attention_bilstm, encode_split(test_tokens, attn_vocab))
attention_metrics = evaluate(y_test, attn_probs)
print("Attention BiLSTM v1 test:", attention_metrics)


## 6) Results on the canonical v1 test split

All models share one held-out split (also used by the classical ML models and
`training/train_lstm.py`), so the numbers are directly comparable with
`models.json`.


In [ ]:
v1_results = {
    "bilstm": bilstm_metrics,
    "textcnn": textcnn_metrics,
    "attention_bilstm": attention_metrics,
}
for model_id, m in v1_results.items():
    c = m["confusion"]
    print(
        f"{model_id:18s} acc={m['accuracy']:.4f} f1={m['f1']:.4f} "
        f"roc_auc={m['roc_auc']:.4f} ece={m['ece']:.4f} "
        f"confusion tn={c['tn']} fp={c['fp']} fn={c['fn']} tp={c['tp']}"
    )


## 7) Sanity check on v2: cross-domain condition

The models were trained on v1; `dataset_v2.parquet` is a separate collection.
The `cross_domain__news` test condition holds out human **news** text (a human
domain the models never saw) against AI-generated text, so it is a quick
generalization signal rather than a headline number. A deterministic balanced
subsample keeps the CPU cost low; tokenization reuses the project's shekar
pipeline (`aitextd.text.tokenize_persian_text`).


In [ ]:

from aitextd.text import tokenize_persian_text

SUBSAMPLE_PER_CLASS = 600

df_v2 = pd.read_parquet(DATASET_V2_PARQUET).reset_index(drop=True)
splits_v2 = joblib.load(SPLITS_V2_JOBLIB)
v2_idx = np.asarray(splits_v2["cross_domain__news__test"])
labels_v2 = df_v2["label_binary"].to_numpy()

rng = np.random.default_rng(RANDOM_SEED)
neg_pool, pos_pool = v2_idx[labels_v2[v2_idx] == 0], v2_idx[labels_v2[v2_idx] == 1]
keep = np.concatenate([
    rng.choice(neg_pool, size=min(SUBSAMPLE_PER_CLASS, len(neg_pool)), replace=False),
    rng.choice(pos_pool, size=min(SUBSAMPLE_PER_CLASS, len(pos_pool)), replace=False),
])
docs_v2 = df_v2.iloc[keep]
y_v2 = docs_v2["label_binary"].to_numpy()
tokens_v2 = [tokenize_persian_text(str(t)) for t in docs_v2["text"]]
print(f"v2 cross_domain__news sanity set: {len(keep)} docs "
      f"(human={int((y_v2 == 0).sum())}, ai={int((y_v2 == 1).sum())})")

v2_results = {}
for model_id, model, vocab in (
    ("bilstm", bilstm, bilstm_vocab),
    ("textcnn", textcnn, textcnn_vocab),
    ("attention_bilstm", attention_bilstm, attn_vocab),
):
    probs_v2 = predict_probs(model, encode_split(tokens_v2, vocab))
    v2_results[model_id] = evaluate(y_v2, probs_v2)
    m = v2_results[model_id]
    print(f"{model_id:18s} acc={m['accuracy']:.4f} f1={m['f1']:.4f} roc_auc={m['roc_auc']:.4f}")


## 8) Register in `models.json` (guarded)

Builds the two new registry entries in exactly the structure the serving stack
expects (same shape as the `bilstm` entry) from the metrics computed above.
Only runs when `FORCE_RETRAIN` is True, so re-running the notebook never
rewrites the registry by accident.


In [ ]:
def metrics_json(m):
    return {
        "accuracy": m["accuracy"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1Score": m["f1"],
        "rocAuc": m["roc_auc"],
    }


def confusion_json(m):
    c = m["confusion"]
    return {
        "trueNegative": c["tn"], "falsePositive": c["fp"],
        "falseNegative": c["fn"], "truePositive": c["tp"],
    }


def build_entry(model_id, name, name_fa, description, description_fa, algorithm,
                filename, runtime, metrics, confusion, feature_count):
    return {
        "id": model_id,
        "name": name,
        "nameFa": name_fa,
        "description": description,
        "descriptionFa": description_fa,
        "algorithm": algorithm,
        "type": "dl",
        "version": "1.0",
        "filename": filename,
        "runtime": runtime,
        "ensembleWeight": 0.5,
        "metrics": metrics,
        "confusionMatrix": confusion,
        "training": {
            "totalSamples": len(train_idx) + len(test_idx),
            "trainSamples": len(train_idx),
            "testSamples": len(test_idx),
            "featureCount": feature_count,
            "testSplitRatio": 0.2,
            "randomSeed": RANDOM_SEED,
            "dataset": "persian_ai_generated_final.csv + blogs.csv",
        },
    }


ENTRIES = [
    build_entry(
        "textcnn", "TextCNN", "شبکه کانولوشنی متن",
        "A TextCNN classifier with its own learned word embeddings: parallel 1D convolutions "
        "with kernel sizes 3, 4 and 5 act as n-gram detectors over the embedded token sequence, "
        "followed by ReLU, a length-masked global max-pooling per kernel (windows of pure "
        "padding are excluded), dropout and a linear head. Trained from scratch on CPU on the "
        "lemmatized Persian text with epoch selection on the canonical held-out test split, to "
        "distinguish AI-generated content from human writing.",
        "دسته‌بند TextCNN با embedding آموخته‌شده اختصاصی: کانولوشن‌های یک‌بعدی موازی با اندازه‌های ۳ و ۴ و ۵ "
        "مانند آشکارسازهای n-گرام روی توالی توکن‌های همسان‌شده عمل می‌کنند و پس از آن‌ها ReLU، بیشینه‌گیری "
        "سراسری با ماسک طول برای هر هسته (پنجره‌های تمام-padding حذف می‌شوند)، dropout و سر خطی قرار دارد. "
        "از ابتدا روی CPU و روی متن فارسی نرمال‌شده و ریشه‌یابی‌شده با انتخاب دوره بر پایه تفکیک آزمون مرجع "
        "آموزش دیده تا متن تولیدشده توسط هوش مصنوعی را از نوشته انسان جدا کند.",
        "TextCNN", "textcnnModel.pt", "pytorch-textcnn",
        metrics_json(textcnn_metrics), confusion_json(textcnn_metrics), len(textcnn_vocab),
    ),
    build_entry(
        "attention_bilstm", "Attention BiLSTM", "دوسویه LSTM با مکانیزم توجه",
        "A bidirectional LSTM with its own learned word embeddings and additive attention "
        "pooling: a learned score vector produces one attention weight per hidden state, "
        "softmax-normalized with pad positions masked out so the pooled summary is built only "
        "from real tokens, followed by dropout and a linear head. Trained from scratch on CPU "
        "on the lemmatized Persian text with epoch selection on the canonical held-out test "
        "split, to distinguish AI-generated content from human writing.",
        "شبکه عصبی LSTM دوسویه با embedding آموخته‌شده اختصاصی و جمع‌بندی توجه افزایشی: بردار امتیاز "
        "آموخته‌شده برای هر حالت پنهان یک وزن توجه تولید می‌کند که با نرمال‌سازی softmax و پوشاندن "
        "توکن‌های padding، خلاصهٔ توالی فقط از توکن‌های واقعی ساخته می‌شود؛ سپس dropout و سر خطی. "
        "از ابتدا روی CPU و روی متن فارسی نرمال‌شده و ریشه‌یابی‌شده با انتخاب دوره بر پایه تفکیک آزمون "
        "مرجع آموزش دیده تا متن تولیدشده توسط هوش مصنوعی را از نوشته انسان جدا کند.",
        "BiLSTM + additive attention pooling", "attnBilstmModel.pt", "pytorch-bilstm-attn",
        metrics_json(attention_metrics), confusion_json(attention_metrics), len(attn_vocab),
    ),
]

if FORCE_RETRAIN:
    import json

    registry = json.loads(MODELS_JSON.read_text(encoding="utf-8"))
    registry = [m for m in registry if m["id"] not in {"textcnn", "attention_bilstm"}]
    after_bilstm = next(i for i, m in enumerate(registry) if m["id"] == "bilstm") + 1
    registry[after_bilstm:after_bilstm] = ENTRIES
    MODELS_JSON.write_text(
        json.dumps(registry, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print("models.json updated with:", [e["id"] for e in ENTRIES])
else:
    print("skipped (FORCE_RETRAIN = False)")


## 9) Summary

One row per model and evaluation split; the v2 rows are the cross-domain
sanity numbers, not comparable with the v1 test rows.


In [ ]:
METRIC_COLS = ("accuracy", "precision", "recall", "f1", "roc_auc", "ece")

summary = pd.DataFrame(
    [{"split": "v1 test", "model": model_id, **{k: m[k] for k in METRIC_COLS}}
     for model_id, m in v1_results.items()]
    + [{"split": "v2 cross_domain__news (sanity)", "model": model_id, **{k: m[k] for k in METRIC_COLS}}
       for model_id, m in v2_results.items()]
)
summary
